In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.sklearn.processing import ScriptProcessor
from sagemaker.spark.processing import PySparkProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import CacheConfig

In [ ]:
# 1. Configuración inicial
role = get_execution_role()
sagemaker_session = sagemaker.Session()
default_bucket = sagemaker_session.default_bucket()
cache_config = CacheConfig(enable_caching=True, expire_after="P3D")

# URIs en S3
uri_script_limpieza = f"s3://{default_bucket}/scripts/PA/TEST_PS_PA_1_limpieza.py"
uri_script_modelado = f"s3://{default_bucket}/scripts/PA/TEST_PS_PA_2_modelado.py"
uri_script_reglas = f"s3://{default_bucket}/scripts/PA/TEST_PS_PA_3_reglas_negocio.py"

# --- PASO 1: LIMPIEZA ---
processor_limpieza = ScriptProcessor(
    command=['python3'],
    image_uri=sagemaker.image_uris.retrieve("sklearn", sagemaker_session.boto_region_name, "1.2-1"),
    role=role,
    instance_count=1,
    instance_type="ml.m5.2xlarge"
)

step_limpieza = ProcessingStep(
    name="PS-Panama-Paso1-Limpieza",
    processor=processor_limpieza,
    code=uri_script_limpieza,
    outputs=[
        ProcessingOutput(
            output_name="output_limpieza",
            source="/opt/ml/processing/output/limpieza"
        )
    ]
)

In [ ]:
# --- PASO 2: MODELADO (ALS CON PYSPARK) ---
processor_modelado = PySparkProcessor(
    base_job_name="sm-spark-als-pa",
    framework_version="3.3",
    role=role,
    instance_count=1,
    instance_type="ml.m5.2xlarge",
    max_runtime_in_seconds=7200,
    env={"PYTHONPATH": "/opt/ml/processing/input/code"}
)

step_modelado = ProcessingStep(
    name="PS-Panama-Paso2-Modelado",
    processor=processor_modelado,
    code=uri_script_modelado,
    inputs=[
        ProcessingInput(
            source=step_limpieza.properties.ProcessingOutputConfig.Outputs["output_limpieza"].S3Output.S3Uri,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="output_modelado",
            source="/opt/ml/processing/output/modelado"
        )
    ]
)

In [ ]:
# --- PASO 3: REGLAS DE NEGOCIO ---
processor_reglas = ScriptProcessor(
    command=['python3'],
    image_uri=sagemaker.image_uris.retrieve("sklearn", sagemaker_session.boto_region_name, "1.2-1"),
    role=role,
    instance_count=1,
    instance_type="ml.m5.2xlarge"
)

step_reglas = ProcessingStep(
    name="PS-Panama-Paso3-ReglasNegocio",
    processor=processor_reglas,
    code=uri_script_reglas,
    inputs=[
        ProcessingInput(
            source=step_limpieza.properties.ProcessingOutputConfig.Outputs["output_limpieza"].S3Output.S3Uri,
            destination="/opt/ml/processing/input/limpieza"
        ),
        ProcessingInput(
            source=step_modelado.properties.ProcessingOutputConfig.Outputs["output_modelado"].S3Output.S3Uri,
            destination="/opt/ml/processing/input/modelado"
        )
    ]
)

In [ ]:
# --- DEFINICIÓN Y CREACIÓN DEL PIPELINE ---
pipeline = Pipeline(
    name="Pipeline-PedidoSugerido-Panama",
    steps=[step_limpieza, step_modelado, step_reglas],
    sagemaker_session=sagemaker_session
)

pipeline.upsert(role_arn=role)
print("Pipeline registrado con éxito.")

In [ ]:
# Descomenta la siguiente línea si deseas forzar una ejecución manual ahora mismo:
# execution = pipeline.start()
# execution.wait()